In [1]:
# Imports, fixed seed, and project paths. The seed is set here so that every random
# operation later in the notebook is reproducible from a single point.
import json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
import shap
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)
warnings.filterwarnings("ignore", category=FutureWarning)

# Resolve the project root whether the notebook runs from /notebooks or from the repo root.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PROC = ROOT / "data" / "processed"
OUT_DIR = ROOT / "outputs"

print("ROOT:", ROOT)
print("xgboost", xgb.__version__, "| shap", shap.__version__, "| pandas", pd.__version__)

# List the artefact directories before loading anything. Filenames are confirmed by
# inspection rather than assumed, so that a wrong path fails here and not silently later.
for d in [DATA_PROC, OUT_DIR / "models", OUT_DIR / "shap", OUT_DIR / "tables"]:
    print(f"\n--- {d} ---")
    if d.exists():
        for f in sorted(d.iterdir()):
            print(f"  {f.name:45s} {f.stat().st_size:>12,} bytes")
    else:
        print("  MISSING")

c:\Users\chara\anaconda3\envs\graphaml\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ROOT: c:\Fintech-Project\graph_AML_pipeline
xgboost 2.1.4 | shap 0.46.0 | pandas 2.2.3

--- c:\Fintech-Project\graph_AML_pipeline\data\processed ---
  01_audit.parquet                               133,085,307 bytes
  03_split_index.parquet                             169,916 bytes
  04_account_map.parquet                           5,848,579 bytes
  04_graph_train.pkl                              77,912,531 bytes
  05_feature_timings.json                                316 bytes
  05_features_full.parquet                       283,826,948 bytes
  05_imputation_values.json                              572 bytes
  05_vertex_features.parquet                      15,260,078 bytes

--- c:\Fintech-Project\graph_AML_pipeline\outputs\models ---
  06_category_maps.json                                  924 bytes
  06_m1_features.json                                    188 bytes
  06_m1_threshold.json                                   186 bytes
  07_m2_features.json                               

In [2]:
# Confirm the artefact filenames found in cell 2 and load the small metadata files.
# Nothing is computed here: this cell exists so that every key name used later in the
# notebook is read from disk rather than assumed.
FEATURES_PARQUET = DATA_PROC / "05_features_full.parquet"
M1_MODEL         = OUT_DIR / "models" / "m1_baseline.json"
M2_MODEL         = OUT_DIR / "models" / "m2_graph_xgb.json"
M1_THRESH_JSON   = OUT_DIR / "models" / "06_m1_threshold.json"
M2_THRESH_JSON   = OUT_DIR / "models" / "07_m2_threshold.json"
M1_FEATS_JSON    = OUT_DIR / "models" / "06_m1_features.json"
M2_FEATS_JSON    = OUT_DIR / "models" / "07_m2_features.json"
CATEGORY_MAPS    = OUT_DIR / "models" / "06_category_maps.json"

for p in [FEATURES_PARQUET, M1_MODEL, M2_MODEL, M1_THRESH_JSON,
          M2_THRESH_JSON, M1_FEATS_JSON, M2_FEATS_JSON, CATEGORY_MAPS]:
    assert p.exists(), f"NOT FOUND: {p}"

m1_meta = json.loads(M1_THRESH_JSON.read_text())
m2_meta = json.loads(M2_THRESH_JSON.read_text())
print("06_m1_threshold.json :", json.dumps(m1_meta, indent=2))
print("07_m2_threshold.json :", json.dumps(m2_meta, indent=2))

# Feature lists are loaded from the training artefacts rather than retyped, so that the
# column order used for prediction is identical to the order used for fitting. A silent
# column-order mismatch produces valid-looking predictions from the wrong model inputs.
m1_feats_raw = json.loads(M1_FEATS_JSON.read_text())
m2_feats_raw = json.loads(M2_FEATS_JSON.read_text())
print("\n06_m1_features.json  :", json.dumps(m1_feats_raw)[:400])
print("\n07_m2_features.json  :", json.dumps(m2_feats_raw)[:700])

# The two saved probability files carry the scores that H1 was decided on.
p1_df = pd.read_parquet(OUT_DIR / "models" / "m1_test_probs.parquet")
p2_df = pd.read_parquet(OUT_DIR / "models" / "m2_test_probs.parquet")
print(f"\nm1_test_probs: shape {p1_df.shape}, columns {list(p1_df.columns)}")
print(p1_df.head(3))
print(f"\nm2_test_probs: shape {p2_df.shape}, columns {list(p2_df.columns)}")
print(p2_df.head(3))

06_m1_threshold.json : {
  "threshold": 0.9579981881141687,
  "val_f1": 0.12072243346007605,
  "chosen_on": "validation",
  "n_trees_used": 482,
  "num_boost_round": 500,
  "early_stopping_rounds": 50
}
07_m2_threshold.json : {
  "threshold": 0.9533845419883794,
  "val_f1": 0.1520912547528517,
  "chosen_on": "validation",
  "n_trees_used": 430,
  "num_boost_round": 500,
  "early_stopping_rounds": 50
}

06_m1_features.json  : ["Amount Received", "Amount Paid", "From Bank", "To Bank", "Receiving Currency", "Payment Currency", "Payment Format", "hour", "day_of_week", "is_weekend"]

07_m2_features.json  : ["Amount Received", "Amount Paid", "From Bank", "To Bank", "Receiving Currency", "Payment Currency", "Payment Format", "hour", "day_of_week", "is_weekend", "from_g_pagerank", "from_g_in_degree", "from_g_out_degree", "from_g_in_weighted", "from_g_out_weighted", "from_g_clustering", "from_g_net_flow", "from_g_two_hop_out", "to_g_pagerank", "to_g_in_degree", "to_g_out_degree", "to_g_in_weigh

In [3]:
# Establish how the saved probabilities align to the feature frame before joining anything.
# The probability files are indexed from 812708, which is not the position the test split
# would occupy in a temporally sorted frame, so the index is inspected rather than assumed.
# An incorrect join here would attach each explanation to the wrong transaction while
# producing no error, so alignment is verified explicitly.
import pyarrow.parquet as pq

schema = pq.ParquetFile(FEATURES_PARQUET).schema_arrow
cols = list(schema.names)
print(f"05_features_full.parquet: {len(cols)} columns")
print("non-feature columns:", [c for c in cols if c not in m2_feats_raw])

# Load only the columns needed: model features, the label, and any split/flag columns.
keep = [c for c in cols if c in m2_feats_raw] + \
       [c for c in cols if c in ("Is Laundering", "split", "is_main", "Timestamp")]
df = pd.read_parquet(FEATURES_PARQUET, columns=keep)
print(f"\nloaded frame: {df.shape}")
print("index:", df.index.min(), "->", df.index.max(), "| unique:", df.index.is_unique)

# The saved probabilities carry their own index. If that index is a subset of the feature
# frame's index, the two align directly and no positional assumption is needed.
print("\nprob index:", p1_df.index.min(), "->", p1_df.index.max(),
      "| n =", len(p1_df), "| unique:", p1_df.index.is_unique)
print("prob index is subset of frame index:", p1_df.index.isin(df.index).all())
print("m1 and m2 prob indices identical:", p1_df.index.equals(p2_df.index))

# Cross-check the two population counts recorded in Notebook 08.
print(f"\nis_main True: {int(p1_df['is_main'].sum()):,}  (expected 760,531)")
if "split" in df.columns:
    print("split counts:\n", df["split"].value_counts())

05_features_full.parquet: 28 columns
non-feature columns: ['Timestamp', 'Account', 'Account.1', 'Is Laundering', 'split']

loaded frame: (5078345, 26)
index: 0 -> 5078344 | unique: True

prob index: 812708 -> 5078344 | n = 761639 | unique: True
prob index is subset of frame index: True
m1 and m2 prob indices identical: True

is_main True: 760,531  (expected 760,531)
split counts:
 split
train    3554957
val       761749
test      761639
Name: count, dtype: int64


In [4]:
# Identify which model inputs are absent from the stored feature frame and must be
# reconstructed. Three of M2's twenty-six features are derived from the timestamp at
# fitting time and were not persisted, so they are rebuilt here rather than assumed.
missing = [f for f in m2_feats_raw if f not in df.columns]
print("features absent from the parquet:", missing)

# Inspect the timestamp and the categorical columns. XGBoost was trained on integer codes,
# so if these columns are still stored as text they must be encoded using the saved maps
# before prediction; encoding them any other way would attribute values to the wrong
# categories without raising an error.
print("\nTimestamp dtype:", df["Timestamp"].dtype)
print(df["Timestamp"].head(3).to_string())

for c in ["Receiving Currency", "Payment Currency", "Payment Format"]:
    print(f"\n{c}: dtype {df[c].dtype}, {df[c].nunique()} unique")
    print("  sample values:", df[c].dropna().unique()[:5])

# The category maps saved by Notebook 06 are the authoritative encoding. Notebook 08
# verified the code counts as 15 / 15 / 7.
cat_maps = json.loads(CATEGORY_MAPS.read_text())
print("\n06_category_maps.json keys:", list(cat_maps))
for k, v in cat_maps.items():
    print(f"  {k}: {len(v) if hasattr(v, '__len__') else v} entries -> {str(v)[:180]}")

features absent from the parquet: ['hour', 'day_of_week', 'is_weekend']

Timestamp dtype: datetime64[ns]
0   2022-09-01 00:20:00
1   2022-09-01 00:20:00
2   2022-09-01 00:00:00

Receiving Currency: dtype object, 15 unique
  sample values: ['US Dollar' 'Bitcoin' 'Euro' 'Australian Dollar' 'Yuan']

Payment Currency: dtype object, 15 unique
  sample values: ['US Dollar' 'Bitcoin' 'Euro' 'Australian Dollar' 'Yuan']

Payment Format: dtype object, 7 unique
  sample values: ['Reinvestment' 'Cheque' 'Credit Card' 'ACH' 'Cash']

06_category_maps.json keys: ['Receiving Currency', 'Payment Currency', 'Payment Format']
  Receiving Currency: 15 entries -> {'0': 'Australian Dollar', '1': 'Bitcoin', '2': 'Brazil Real', '3': 'Canadian Dollar', '4': 'Euro', '5': 'Mexican Peso', '6': 'Ruble', '7': 'Rupee', '8': 'Saudi Riyal', '9': 'Sheke
  Payment Currency: 15 entries -> {'0': 'Australian Dollar', '1': 'Bitcoin', '2': 'Brazil Real', '3': 'Canadian Dollar', '4': 'Euro', '5': 'Mexican Peso', '6': 'Ruble',

In [5]:
# Reconstruct the three derived time features and encode the categorical columns using
# the maps saved at training time. The reconstruction is then verified by re-scoring the
# test split and comparing against the probabilities stored when the models were fitted.
# This check is the reason the notebook can claim to explain the models that H1 was
# decided on, rather than a re-created approximation of them.

# Time features, derived from the timestamp exactly as at fitting time.
df["hour"] = df["Timestamp"].dt.hour
df["day_of_week"] = df["Timestamp"].dt.dayofweek        # Monday = 0
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)  # Saturday and Sunday

# Categorical encoding. The saved maps run code -> label, so they are inverted to
# label -> code. Encoding by any other rule (for example re-running a fresh
# factorisation) would assign different integers to the same currencies and cause SHAP
# to attribute values to the wrong categories with no error raised.
for col, code_to_label in cat_maps.items():
    label_to_code = {label: int(code) for code, label in code_to_label.items()}
    unmapped = set(df[col].dropna().unique()) - set(label_to_code)
    assert not unmapped, f"{col}: values absent from the saved map: {unmapped}"
    df[col] = df[col].map(label_to_code).astype("int32")

print("reconstructed:", df[["hour", "day_of_week", "is_weekend"]].dtypes.to_dict())
print("encoded max codes:", {c: int(df[c].max()) for c in cat_maps})  # expect 14, 14, 6

# Attach the stored probabilities and the population flag by index join.
df.loc[p1_df.index, "m1_prob"] = p1_df["m1_prob"].values
df.loc[p2_df.index, "m2_prob"] = p2_df["m2_prob"].values
df.loc[p1_df.index, "is_main"] = p1_df["is_main"].values

test = df.loc[p1_df.index]
print(f"\ntest rows joined: {len(test):,}")

# Verification: re-score the test split with the sliced boosters and compare against the
# stored probabilities. The boosters are sliced to the tree counts the models were scored
# at (482 and 430); the saved files contain more trees than were used.
M1_FEATS, M2_FEATS = m1_feats_raw, m2_feats_raw
m1 = xgb.Booster(); m1.load_model(str(M1_MODEL)); m1 = m1[:m1_meta["n_trees_used"]]
m2 = xgb.Booster(); m2.load_model(str(M2_MODEL)); m2 = m2[:m2_meta["n_trees_used"]]

chk1 = m1.predict(xgb.DMatrix(test[M1_FEATS]))
chk2 = m2.predict(xgb.DMatrix(test[M2_FEATS]))
d1 = np.abs(chk1 - test["m1_prob"].values).max()
d2 = np.abs(chk2 - test["m2_prob"].values).max()
print(f"\nmax abs difference vs stored probabilities -- M1 {d1:.3e}  M2 {d2:.3e}")

assert d1 < 1e-6 and d2 < 1e-6, (
    "reconstruction does not reproduce the stored probabilities -- the feature "
    "reconstruction or the encoding is wrong; do not proceed")
print("Reconstruction verified. The features fed to SHAP are the features the models were scored on.")

reconstructed: {'hour': dtype('int32'), 'day_of_week': dtype('int32'), 'is_weekend': dtype('int32')}
encoded max codes: {'Receiving Currency': 14, 'Payment Currency': 14, 'Payment Format': 6}

test rows joined: 761,639

max abs difference vs stored probabilities -- M1 0.000e+00  M2 0.000e+00
Reconstruction verified. The features fed to SHAP are the features the models were scored on.


In [6]:
# Restrict to the D1 evaluation population (test-main) and construct the four case
# categories registered in the pre-registration as exploratory. Category membership is
# defined by each model's own decision threshold, frozen on the validation split before
# any test data was examined; a conventional 0.5 cut-off would define a different
# operating point from the one every other result in this project reports.
test_main = test[test["is_main"] == True].copy()
y  = test_main["Is Laundering"].values.astype(int)
p1 = test_main["m1_prob"].values
p2 = test_main["m2_prob"].values
T1, T2 = m1_meta["threshold"], m2_meta["threshold"]

print(f"test-main: {len(test_main):,} rows, {y.sum():,} illicit")
print(f"thresholds -- M1 {T1:.6f}  M2 {T2:.6f}")

flag1, flag2 = p1 >= T1, p2 >= T2
correct1, correct2 = flag1 == (y == 1), flag2 == (y == 1)

# Cross-check against the figures reported in Notebook 08b. These must match, or the
# population or threshold in use here differs from the one already reported.
print(f"\nM2 flagged {int(flag2.sum()):,} (expected 2,720), "
      f"true positives {int((flag2 & (y == 1)).sum()):,} (expected 216)")
print(f"M1 flagged {int(flag1.sum()):,}, true positives {int((flag1 & (y == 1)).sum()):,}")

# The four registered categories. Category 4 is reported in two forms: the illicit subset,
# in which M1 detected a laundering transaction that M2 subsequently missed, is preferred
# because it expresses the H1 result at the level of one transaction; the broader form is
# retained as a fallback should the illicit subset be empty.
cases = {
    "1_both_correct":       (y == 1) & correct1 & correct2,
    "2_m2_fixes_fn":        (y == 1) & ~flag1 & flag2,
    "3_m2_fixes_fp":        (y == 0) & flag1 & ~flag2,
    "4_m2_breaks_illicit":  (y == 1) & flag1 & ~flag2,
    "4b_m2_breaks_any":     correct1 & ~correct2,
}
counts = pd.Series({k: int(v.sum()) for k, v in cases.items()}, name="candidates")
print("\ncandidate counts per category:")
print(counts.to_string())

test-main: 760,531 rows, 906 illicit
thresholds -- M1 0.957998  M2 0.953385

M2 flagged 2,720 (expected 2,720), true positives 216 (expected 216)
M1 flagged 3,646, true positives 291

candidate counts per category:
1_both_correct          121
2_m2_fixes_fn            95
3_m2_fixes_fp          2859
4_m2_breaks_illicit     170
4b_m2_breaks_any       2178


In [7]:
# Select one representative transaction per category. The rule is fixed in advance: within
# each category, take the transaction on which the two models disagree most, measured by
# the absolute difference in predicted probability, with ties broken by the lowest index.
# The rule is applied before any attribution is computed, so the cases cannot have been
# chosen for the explanations they happen to produce. Maximum disagreement is the natural
# criterion because these figures exist to show where the two models diverge.
CATEGORIES = ["1_both_correct", "2_m2_fixes_fn", "3_m2_fixes_fp", "4_m2_breaks_illicit"]

gap = np.abs(p2 - p1)
idx_values = test_main.index.values

picks = {}
for c in CATEGORIES:
    mask = cases[c]
    if mask.sum() == 0:
        print(f"{c}: EMPTY -- record this in the report")
        continue
    pos = np.where(mask)[0]
    # Sort by descending disagreement, then ascending index, and take the first.
    chosen = pos[np.lexsort((idx_values[pos], -gap[pos]))[0]]
    picks[c] = int(chosen)

sel_pos = np.array([picks[c] for c in CATEGORIES if c in picks])
sel_idx = idx_values[sel_pos]

summary = pd.DataFrame({
    "case": [c for c in CATEGORIES if c in picks],
    "df_index": sel_idx,
    "is_laundering": y[sel_pos],
    "m1_prob": p1[sel_pos].round(6),
    "m2_prob": p2[sel_pos].round(6),
    "abs_gap": gap[sel_pos].round(6),
    "m1_flags": flag1[sel_pos],
    "m2_flags": flag2[sel_pos],
})
print(summary.to_string(index=False))

# Show the transaction-level detail of each selected case, so the report can describe
# what kind of payment each explanation concerns rather than only its index.
code_to_label = {c: {int(k): v for k, v in m.items()} for c, m in cat_maps.items()}
detail = test_main.loc[sel_idx, ["Timestamp", "Amount Paid", "Amount Received",
                                 "Payment Format", "Payment Currency", "From Bank", "To Bank"]].copy()
detail["Payment Format"] = detail["Payment Format"].map(code_to_label["Payment Format"])
detail["Payment Currency"] = detail["Payment Currency"].map(code_to_label["Payment Currency"])
detail.insert(0, "case", [c for c in CATEGORIES if c in picks])
print("\n", detail.to_string(index=False))

               case  df_index  is_laundering  m1_prob  m2_prob  abs_gap  m1_flags  m2_flags
     1_both_correct   3732458              1 0.995578 0.953871 0.041707      True      True
      2_m2_fixes_fn   2655559              1 0.228883 0.990833 0.761950     False      True
      3_m2_fixes_fp   5022147              0 0.988940 0.022487 0.966453      True     False
4_m2_breaks_illicit   2313434              1 0.961138 0.083541 0.877597      True     False

                case           Timestamp  Amount Paid  Amount Received Payment Format  Payment Currency  From Bank  To Bank
     1_both_correct 2022-09-10 13:04:00     14655.44         14655.44            ACH         US Dollar      17854     1467
      2_m2_fixes_fn 2022-09-09 19:20:00   9667383.73       9667383.73            ACH              Euro      21749     1502
      3_m2_fixes_fp 2022-09-10 16:34:00     63031.55         63031.55            ACH       Saudi Riyal     148348   148348
4_m2_breaks_illicit 2022-09-10 13:17:00      2

In [8]:
# Compute interventional TreeSHAP attributions for the four selected transactions under
# both models. The background distribution is rebuilt exactly as in the global analysis:
# one thousand training rows drawn with the same seed and stratified on the outcome, so
# that local attributions are on the same scale as the global ones already reported.
# Interventional attributions are used rather than the path-dependent variant because they
# answer a causal question about the model's own computation (Janzing et al., 2020), which
# is the semantics required when an explanation must be defended to a regulator.
train = df[df["split"] == "train"]
n_bg = 1000
n_pos = max(1, int(n_bg * train["Is Laundering"].mean()))
bg = pd.concat([
    train[train["Is Laundering"] == 1].sample(n_pos, random_state=SEED),
    train[train["Is Laundering"] == 0].sample(n_bg - n_pos, random_state=SEED),
]).sample(frac=1, random_state=SEED)
print(f"background: {len(bg)} rows, {int(bg['Is Laundering'].sum())} illicit")

X1 = test_main.loc[sel_idx, M1_FEATS]
X2 = test_main.loc[sel_idx, M2_FEATS]

expl1 = shap.TreeExplainer(m1, data=bg[M1_FEATS],
                           feature_perturbation="interventional", model_output="raw")
expl2 = shap.TreeExplainer(m2, data=bg[M2_FEATS],
                           feature_perturbation="interventional", model_output="raw")
sv1 = expl1.shap_values(X1)
sv2 = expl2.shap_values(X2)

print(f"\nbase values -- M1 {expl1.expected_value:.6f}  M2 {expl2.expected_value:.6f}")
print("Notebook 08 recorded -6.121312 and -6.330573")

# Additivity check. Each attribution vector, added to the model's base value, must
# reproduce the raw margin the model actually output for that transaction. This confirms
# the explanations account for the whole prediction and none of it is unattributed.
marg1 = m1.predict(xgb.DMatrix(X1), output_margin=True)
marg2 = m2.predict(xgb.DMatrix(X2), output_margin=True)
g1 = np.abs(expl1.expected_value + sv1.sum(axis=1) - marg1).max()
g2 = np.abs(expl2.expected_value + sv2.sum(axis=1) - marg2).max()
print(f"max additivity gap -- M1 {g1:.3e}  M2 {g2:.3e}")
assert g1 < 1e-3 and g2 < 1e-3, "additivity violated -- do not use these attributions"
print("Additivity holds.")

background: 1000 rows, 1 illicit

base values -- M1 -6.121312  M2 -6.330573
Notebook 08 recorded -6.121312 and -6.330573
max additivity gap -- M1 2.342e-06  M2 1.951e-06
Additivity holds.


In [9]:
# Tabulate the leading attributions for each selected transaction under both models.
# Values are on the log-odds scale, so they are directly comparable with the base value:
# a positive contribution pushes the prediction towards "laundering", a negative one away.
# The point of the comparison is not which model scores higher but which evidence each
# model relies on when explaining the same transaction.
def top_k(row, feats, k=6):
    order = np.argsort(-np.abs(row))[:k]
    return [(feats[i], float(row[i])) for i in order]

for j, c in enumerate([c for c in CATEGORIES if c in picks]):
    pos = picks[c]
    print(f"\n{'='*78}\n{c}  |  index {sel_idx[j]}  |  y={y[pos]}  "
          f"|  M1 {p1[pos]:.4f} {'FLAG' if flag1[pos] else 'pass'}  "
          f"|  M2 {p2[pos]:.4f} {'FLAG' if flag2[pos] else 'pass'}")
    print(f"{'M1 (transaction features only)':<44}{'M2 (with graph features)':<44}")
    t1, t2 = top_k(sv1[j], M1_FEATS), top_k(sv2[j], M2_FEATS)
    for a, b in zip(t1, t2):
        print(f"  {a[0][:28]:<28}{a[1]:>+9.4f}     {b[0][:28]:<28}{b[1]:>+9.4f}")
    # Share of the explanation carried by graph features in M2, for this transaction.
    g = sum(abs(v) for f, v in zip(M2_FEATS, sv2[j]) if f.startswith(("from_g_", "to_g_")))
    tot = np.abs(sv2[j]).sum()
    print(f"  graph-feature share of M2's explanation for this transaction: {100*g/tot:.1f}%")
    


1_both_correct  |  index 3732458  |  y=1  |  M1 0.9956 FLAG  |  M2 0.9539 FLAG
M1 (transaction features only)              M2 (with graph features)                    
  Payment Format                +6.2194     Payment Format                +5.5319
  Amount Paid                   +1.6291     Amount Paid                   +1.1619
  Amount Received               +1.2456     Amount Received               +1.0319
  day_of_week                   +0.5954     from_g_pagerank               +0.5056
  To Bank                       +0.4600     from_g_net_flow               +0.4684
  hour                          +0.4069     from_g_two_hop_out            -0.4677
  graph-feature share of M2's explanation for this transaction: 27.5%

2_m2_fixes_fn  |  index 2655559  |  y=1  |  M1 0.2289 pass  |  M2 0.9908 FLAG
M1 (transaction features only)              M2 (with graph features)                    
  Payment Format                +5.3089     Payment Format                +5.5275
  Amount Received  

In [10]:
# Produce a paired waterfall plot for each selected transaction: the same transaction
# explained by each model, drawn on the same log-odds scale so the two are directly
# comparable. Only one pair appears in the report, because the assessment's word limit
# counts figures; the remaining pairs are retained in the repository as evidence.
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

case_titles = {
    "1_both_correct":      "Both models correct",
    "2_m2_fixes_fn":       "M2 corrects an M1 false negative",
    "3_m2_fixes_fp":       "M2 corrects an M1 false positive",
    "4_m2_breaks_illicit": "M1 correct, M2 wrong",
}

for j, c in enumerate([c for c in CATEGORIES if c in picks]):
    pos = picks[c]
    for tag, sv, feats, X, ev, prob in [
        ("M1", sv1, M1_FEATS, X1, expl1.expected_value, p1[pos]),
        ("M2", sv2, M2_FEATS, X2, expl2.expected_value, p2[pos]),
    ]:
        e = shap.Explanation(values=sv[j], base_values=ev,
                             feature_names=feats, data=X.iloc[j].values)
        shap.plots.waterfall(e, show=False, max_display=10)
        plt.title(f"{case_titles[c]} — {tag}\ntransaction {sel_idx[j]}, "
                  f"p={prob:.4f}, actual={'laundering' if y[pos]==1 else 'legitimate'}",
                  fontsize=9)
        plt.savefig(FIG_DIR / f"09_{c}_{tag.lower()}.png", dpi=300, bbox_inches="tight")
        plt.close()

print(f"saved {2*len(picks)} waterfall plots to {FIG_DIR}")
for f in sorted(FIG_DIR.glob("09_*.png")):
    print("  ", f.name)

saved 8 waterfall plots to c:\Fintech-Project\graph_AML_pipeline\outputs\figures
   09_1_both_correct_m1.png
   09_1_both_correct_m2.png
   09_2_m2_fixes_fn_m1.png
   09_2_m2_fixes_fn_m2.png
   09_3_m2_fixes_fp_m1.png
   09_3_m2_fixes_fp_m2.png
   09_4_m2_breaks_illicit_m1.png
   09_4_m2_breaks_illicit_m2.png


In [11]:
# Establish whether the zero-valued graph features on the selected transactions represent
# genuinely inactive accounts or accounts absent from the training-window graph, whose
# features were imputed under the registered missing-data policy (median for continuous
# features, zero for counts). The distinction matters: the first would be a property of
# the transaction, the second a property of the imputation rule.
imp = json.loads((DATA_PROC / "05_imputation_values.json").read_text())
print("imputation values:", json.dumps(imp, indent=2)[:600])

acct_map = pd.read_parquet(DATA_PROC / "04_account_map.parquet")
print("\naccount map columns:", list(acct_map.columns), "| rows:", f"{len(acct_map):,}")

# For each selected transaction, report whether its sending and receiving accounts appear
# in the training graph, and the raw values of the graph features that drove the outcome.
acct_cols = [c for c in test_main.columns if c in ("Account", "Account.1")]
print("\naccount columns on the frame:", acct_cols)

probe = ["to_g_in_degree", "to_g_out_degree", "from_g_two_hop_out",
         "from_g_out_weighted", "from_g_pagerank", "to_g_in_weighted"]
view = test_main.loc[sel_idx, acct_cols + probe].copy()
view.insert(0, "case", [c for c in CATEGORIES if c in picks])
print("\n", view.to_string())

# How common is a zero receiver in-degree across test-main, and does it relate to outcome?
z = test_main["to_g_in_degree"] == 0
print(f"\nto_g_in_degree == 0: {int(z.sum()):,} of {len(test_main):,} rows ({100*z.mean():.2f}%)")
print(f"  illicit rate where zero:     {100*y[z.values].mean():.4f}%")
print(f"  illicit rate where non-zero: {100*y[~z.values].mean():.4f}%")
print(f"  M2 flags where zero:     {100*flag2[z.values].mean():.4f}%")
print(f"  M2 flags where non-zero: {100*flag2[~z.values].mean():.4f}%")

imputation values: {
  "basis": "median across training accounts, each counted once (decision b, 2026-08-26)",
  "account_basis_applied": {
    "g_pagerank": 1.3446550120393992e-06,
    "g_in_weighted": 12302.345,
    "g_out_weighted": 8657.125,
    "g_clustering": 0.0,
    "g_net_flow": 0.0
  },
  "transaction_basis_rejected": {
    "g_pagerank": 9.98488857414928e-07,
    "g_in_weighted": 180635.15,
    "g_out_weighted": 170169.39000000007,
    "g_clustering": 0.0,
    "g_net_flow": 0.9225720000000001
  },
  "count_features_filled_with": 0,
  "rows_affected": 3030
}

account map columns: ['account_id', 'vertex_index'] | rows: 513,284

account columns on the frame: []

                         case  to_g_in_degree  to_g_out_degree  from_g_two_hop_out  from_g_out_weighted  from_g_pagerank  to_g_in_weighted
3732458       1_both_correct             1.0              4.0                 0.0               656.14     3.884370e-06          30370.63
2655559        2_m2_fixes_fn             1.0 

In [12]:
# Quantify detection performance separately for transactions into accounts with no
# recorded incoming edges in the training graph, and for all other transactions. The
# comparison establishes whether the blind spot exposed by the case-4 explanation is
# introduced by the graph features or merely left uncorrected by them.
zero_in = (test_main["to_g_in_degree"] == 0).values

rows = []
for name, m in [("zero in-degree receiver", zero_in), ("all other rows", ~zero_in)]:
    n, pos = int(m.sum()), int(y[m].sum())
    for label, f in [("M1", flag1), ("M2", flag2)]:
        tp = int((f & m & (y == 1)).sum())
        fl = int((f & m).sum())
        rows.append({
            "stratum": name, "model": label, "n": n, "illicit": pos,
            "illicit_rate_%": round(100 * pos / n, 4),
            "flagged": fl, "true_pos": tp,
            "recall_%": round(100 * tp / pos, 2) if pos else np.nan,
            "precision_%": round(100 * tp / fl, 2) if fl else np.nan,
        })

strata = pd.DataFrame(rows)
print(strata.to_string(index=False))

# Express the same comparison as the share of each model's detections and misses that
# fall in the high-risk stratum.
for label, f in [("M1", flag1), ("M2", flag2)]:
    miss = (~f) & (y == 1)
    print(f"\n{label}: {int((f & (y==1)).sum())} true positives, "
          f"{int((f & (y==1) & zero_in).sum())} of them in the zero-in-degree stratum")
    print(f"    {int(miss.sum())} illicit missed, "
          f"{int((miss & zero_in).sum())} of them in that stratum "
          f"({100*(miss & zero_in).sum()/miss.sum():.1f}% of all misses)")

strata.to_csv(OUT_DIR / "tables" / "09_zero_indegree_strata.csv", index=False)

                stratum model      n  illicit  illicit_rate_%  flagged  true_pos  recall_%  precision_%
zero in-degree receiver    M1  13424      219          1.6314      955        76     34.70         7.96
zero in-degree receiver    M2  13424      219          1.6314        4         1      0.46        25.00
         all other rows    M1 747107      687          0.0920     2691       215     31.30         7.99
         all other rows    M2 747107      687          0.0920     2716       215     31.30         7.92

M1: 291 true positives, 76 of them in the zero-in-degree stratum
    615 illicit missed, 143 of them in that stratum (23.3% of all misses)

M2: 216 true positives, 1 of them in the zero-in-degree stratum
    690 illicit missed, 218 of them in that stratum (31.6% of all misses)


In [13]:
# Test whether the detection failure is specific to a zero in-degree or reflects a smooth
# relationship with receiver in-degree, and whether the stratum is confounded with the
# dominant Payment Format effect. A finding that survives both checks can be reported as
# a property of the graph feature rather than an artefact of how the stratum was defined.
bins = [-0.1, 0.5, 1.5, 3.5, 10.5, np.inf]
labels = ["0", "1", "2-3", "4-10", "11+"]
grp = pd.cut(test_main["to_g_in_degree"], bins=bins, labels=labels)

prof = pd.DataFrame({
    "n": grp.value_counts().reindex(labels),
    "illicit": pd.Series(y, index=test_main.index).groupby(grp, observed=False).sum().reindex(labels),
    "m1_recall_%": pd.Series(flag1 & (y == 1), index=test_main.index).groupby(grp, observed=False).sum().reindex(labels),
    "m2_recall_%": pd.Series(flag2 & (y == 1), index=test_main.index).groupby(grp, observed=False).sum().reindex(labels),
})
prof["m1_recall_%"] = (100 * prof["m1_recall_%"] / prof["illicit"]).round(2)
prof["m2_recall_%"] = (100 * prof["m2_recall_%"] / prof["illicit"]).round(2)
prof["illicit_rate_%"] = (100 * prof["illicit"] / prof["n"]).round(4)
print("Recall by receiver in-degree band:\n")
print(prof.to_string())

# Is the zero-in-degree stratum simply a different mix of payment formats?
fmt = test_main["Payment Format"].map(code_to_label["Payment Format"])
mix = pd.crosstab(fmt, zero_in, normalize="columns").mul(100).round(2)
mix.columns = ["non-zero in-degree %", "zero in-degree %"]
print("\nPayment Format mix by stratum:\n")
print(mix.to_string())

Recall by receiver in-degree band:

                     n  illicit  m1_recall_%  m2_recall_%  illicit_rate_%
to_g_in_degree                                                           
0                13424      219        34.70         0.46          1.6314
1               117069      235        32.34        38.30          0.2007
2-3             555472      382        26.70        22.77          0.0688
4-10             71953       66        53.03        54.55          0.0917
11+               2613        4        50.00        50.00          0.1531

Payment Format mix by stratum:

                non-zero in-degree %  zero in-degree %
Payment Format                                        
ACH                            12.87             77.82
Bitcoin                         2.75              6.19
Cash                           10.95              1.10
Cheque                         41.46              5.48
Credit Card                    28.35              8.81
Wire                        

In [14]:
# Persist the selection decisions, the case-level evidence and the stratified finding so
# that every claim in the report can be traced to a stored artefact, and record explicitly
# which analyses were pre-registered and which were not.
f9_rows = []
for j, c in enumerate([c for c in CATEGORIES if c in picks]):
    pos = picks[c]
    f9_rows.append({
        "case": c, "df_index": int(sel_idx[j]), "is_laundering": int(y[pos]),
        "m1_prob": float(p1[pos]), "m2_prob": float(p2[pos]),
        "m1_top5": " | ".join(f for f, _ in top_k(sv1[j], M1_FEATS, 5)),
        "m2_top5": " | ".join(f for f, _ in top_k(sv2[j], M2_FEATS, 5)),
        "m2_graph_share_%": round(100 * sum(abs(v) for f, v in zip(M2_FEATS, sv2[j])
                                  if f.startswith(("from_g_", "to_g_"))) / np.abs(sv2[j]).sum(), 2),
    })
pd.DataFrame(f9_rows).to_csv(OUT_DIR / "tables" / "09_case_evidence.csv", index=False)

manifest = {
    "notebook": "09_shap_local_paired",
    "date": pd.Timestamp.now().isoformat(timespec="seconds"),
    "registration_status": "exploratory throughout; cannot alter H1-H4",
    "selection_form": "four-category (OSF template line 225); 3-rung ladder retired 8 Sep 2026",
    "selection_rule": "max |p2 - p1| within category, ties by lowest index, fixed before SHAP computed",
    "population": "test-main (is_main), 760531 rows, 906 illicit",
    "reconstruction_check": "sliced boosters reproduce stored probabilities, max abs diff 0.000e+00",
    "thresholds": {"m1": T1, "m2": T2},
    "n_trees": {"m1": m1_meta["n_trees_used"], "m2": m2_meta["n_trees_used"]},
    "base_values": {"m1": float(expl1.expected_value), "m2": float(expl2.expected_value)},
    "candidate_counts": {k: int(v.sum()) for k, v in cases.items()},
    "selected": {c: int(sel_idx[i]) for i, c in enumerate([c for c in CATEGORIES if c in picks])},
    "post_hoc_stratification": {
        "note": "NOT registered; arose from the case-4 explanation during this session",
        "definition": "to_g_in_degree == 0 in test-main",
        "n": 13424, "illicit": 219, "illicit_rate_pct": 1.6314,
        "m1_recall_pct": 34.70, "m2_recall_pct": 0.46,
        "outside_stratum_recall_pct": {"m1": 31.30, "m2": 31.30},
        "confound_declared": "stratum is 77.8% ACH vs 12.9% elsewhere; M1 serves as control",
    },
}
(OUT_DIR / "shap" / "09_local_manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest["post_hoc_stratification"], indent=2))
print("\nartefacts written: 09_case_evidence.csv, 09_zero_indegree_strata.csv, 09_local_manifest.json")

{
  "note": "NOT registered; arose from the case-4 explanation during this session",
  "definition": "to_g_in_degree == 0 in test-main",
  "n": 13424,
  "illicit": 219,
  "illicit_rate_pct": 1.6314,
  "m1_recall_pct": 34.7,
  "m2_recall_pct": 0.46,
  "outside_stratum_recall_pct": {
    "m1": 31.3,
    "m2": 31.3
  },
  "confound_declared": "stratum is 77.8% ACH vs 12.9% elsewhere; M1 serves as control"
}

artefacts written: 09_case_evidence.csv, 09_zero_indegree_strata.csv, 09_local_manifest.json
